# FIRMS quickstart — recent active fires

Fetch recent **active-fire detections** (MODIS / VIIRS) from [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/) through the `earthlens` FIRMS backend, and plot the fire pixels coloured by fire radiative power (FRP).

FIRMS is a `vector` backend: `download()` returns a pyramids `FeatureCollection` (a `geopandas.GeoDataFrame`, CRS `EPSG:4326`), one row per fire pixel.

In [ ]:
import datetime as dt
import os
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('firms_output')
OUT_DIR.mkdir(exist_ok=True)

# FIRMS needs a free MAP_KEY (https://firms.modaps.eosdis.nasa.gov/api/map_key/).
# Set FIRMS_MAP_KEY in your environment; the live cells below skip cleanly
# (nbval-lax-safe) when it is absent, so the notebook never fails offline.
HAS_KEY = bool(os.environ.get('FIRMS_MAP_KEY'))
print('FIRMS_MAP_KEY set:', HAS_KEY)


## Query

`variables` is a list of FIRMS **sensor codes** (not data variables). We use the 375 m VIIRS S-NPP near-real-time sensor over Southern California for a recent 7-day window. FIRMS caps requests at 10 days / one sensor, but the backend chunks longer windows transparently. The result is written to `firms_output/` and returned in memory.

In [ ]:
# A recent 7-day window (FIRMS NRT retains only ~2 months).
TODAY = dt.date.today()
START = (TODAY - dt.timedelta(days=7)).strftime('%Y-%m-%d')
END = TODAY.strftime('%Y-%m-%d')

fires = None
if HAS_KEY:
    try:
        fires = EarthLens(
            data_source='firms',
            variables=['VIIRS_SNPP_NRT'],
            start=START,
            end=END,
            lat_lim=[33.0, 35.0],
            lon_lim=[-119.0, -117.0],
            path=str(OUT_DIR),
        ).download(progress_bar=False)
        print('detections:', len(fires))
    except Exception as exc:  # keep the notebook nbval-lax-safe
        print('skipped live query:', exc)


## Inspect the detections

Every row is one fire pixel with a normalised `confidence_pct` and `frp` (MW).

In [ ]:
if fires is not None and len(fires):
    cols = ['acq_datetime', 'sensor', 'confidence', 'confidence_pct',
            'brightness_k', 'frp', 'daynight', 'geometry']
    display(fires[cols].head())
    print('CRS:', fires.crs)


## Plot the fire pixels coloured by FRP

In [ ]:
if fires is not None and len(fires):
    import matplotlib.pyplot as plt

    ax = fires.plot(column='frp', cmap='inferno', markersize=18, alpha=0.7, legend=True)
    ax.set_title(f'FIRMS VIIRS detections, {START} to {END} (colour = FRP, MW)')
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')
    plt.show()
